# Thực nghiệm 3.2 — CIFAR-10 / CIFAR-100 trên Kaggle

Notebook điều phối code trong `experiments/exp32_cifar.py`; không chứa lại vòng lặp train.
Lưới đầy đủ: **2 dataset × 3 model × ERM/mixup × số seed**, mỗi run **200 epoch**.
Với `TARGET_SEEDS = [0, 1, 2]` có **36 runs**. Ba seed là lựa chọn cho đồ án;
[mục 3.2 của paper](https://arxiv.org/html/1710.09412v2#S3.SS2) không quy định số seed.

**Cách dùng:**

1. Chọn GPU, bật Internet để tải CIFAR lần đầu, Add Input dataset chứa bản `Source` mới nhất.
2. Sửa ô cấu hình: chọn nhóm chạy trong phiên qua `SESSION_*`. Đặt tất cả `SESSION_* = None`
   nếu muốn xếp hàng toàn bộ lưới. Mặc định chỉ chọn **CIFAR-10 / PreAct ResNet-18 / seed 0**, cả ERM và mixup.
3. Để `RUN_TRAINING = False` khi chỉ xem kế hoạch/gom kết quả; đổi thành `True` để huấn luyện.
   Dùng **Save Version → Save & Run All** để lưu output của phiên.
4. Phiên sau: Add Input output của version trước, điền đường dẫn thư mục `results` vào `RESUME_INPUTS`.
   Có thể thêm nhiều thư mục kết quả từ các máy/phiên khác nhau.

Đầu ra chính: `results/`, `results/report/`, `results/management/`, và `results_logs.zip`.
**Giữ thư mục `results` có `last.pt` để resume**; file zip nhẹ không có trọng số.


In [ ]:
# Toàn bộ phạm vi cần hoàn thành (12 runs cho mỗi seed).
TARGET_SEEDS = [0, 1, 2]

# Nhóm được phép chạy trong phiên này; None = toàn bộ lựa chọn trong lưới.
SESSION_DATASETS = ['cifar10']                 # hoặc ['cifar100'], hoặc None
SESSION_MODELS = ['preact_resnet18']           # wideresnet28_10 / densenet_bc190 / None
SESSION_METHODS = None                        # ['erm'], ['mixup'], hoặc None
SESSION_SEEDS = [0]                            # tập con của TARGET_SEEDS, hoặc None

RUN_TRAINING = False
RUN_TESTS = False                             # tùy chọn: chạy tests/ trước khi train
WORKERS = 4
MAX_SESSION_HOURS = 8.0                        # None = không giới hạn; xem ghi chú bên dưới
HARD_STOP_HOURS = 11.0                         # dừng hẳn run đang chạy (tính từ lúc mở notebook) để kịp lưu output trước mốc 12 giờ của Kaggle
STOP_ON_ERROR = True
REPORT_METRIC = 'best_err'                     # best_err / last_err / median_last10_err

# None: tự tìm thư mục có core/, cifar/, experiments/ trong /kaggle/input.
# Nếu có nhiều bản code, điền chính xác thư mục Source muốn dùng.
SOURCE_INPUT = None                           # '/kaggle/input/mixup-source/Source'
RESUME_INPUTS = [
    # '/kaggle/input/ten-notebook-version-truoc/results',
]


## 1. Môi trường và mã nguồn

Cấu hình train được lấy từ runner: SGD, batch 128, LR 0.1, weight decay 1e-4, không dropout;
ERM α=0 và mixup α=1. Lịch LR được chọn theo từng model. Notebook giữ **FP32**, không bật AMP.
Code dùng **một GPU**. Chỉ dựng model khi đến lượt run đó.

Nếu dùng dữ liệu/code đã gắn vào Kaggle Input, các ô dưới chỉ ghi vào `/kaggle/working`.
Cập nhật dataset `Source` khi sửa code; notebook chép lại bản đã chọn mỗi lần chạy phần chuẩn bị.


In [ ]:
import copy
import csv
import json
import math
import os
import shlex
import signal
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, display

NOTEBOOK_START = time.monotonic()  # mốc cho HARD_STOP_HOURS
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
if not WORK.is_dir() or not INPUT.is_dir():
    raise RuntimeError('Notebook này dành cho Kaggle: cần /kaggle/input và /kaggle/working.')
SRC = WORK / 'Source'
DATA = WORK / 'data'
OUT = WORK / 'results'
MANAGEMENT = OUT / 'management'
MANAGEMENT.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0], '| torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
elif RUN_TRAINING:
    raise RuntimeError('Chưa có GPU. Bật Accelerator GPU trước khi RUN_TRAINING=True.')
else:
    print('Chỉ xem kế hoạch/gom báo cáo: chưa cần GPU.')

required = ('core/mixup.py', 'cifar/models.py', 'experiments/exp32_cifar.py',
            'experiments/exp32_report.py')
if SOURCE_INPUT is None:
    candidates = sorted({p.parent.parent for p in INPUT.rglob('core/mixup.py')
                         if all((p.parent.parent / rel).is_file() for rel in required)})
    if len(candidates) != 1:
        raise RuntimeError('Cần chọn SOURCE_INPUT: tìm thấy các bản Source: ' + str(candidates))
    source_input = candidates[0]
else:
    source_input = Path(SOURCE_INPUT).resolve()
    if not source_input.is_relative_to(INPUT.resolve()):
        raise ValueError('SOURCE_INPUT phải nằm trong /kaggle/input.')
    if not all((source_input / rel).is_file() for rel in required):
        raise FileNotFoundError('Source thiếu runner/report hoặc core/cifar: ' + str(source_input))
# Đồng bộ file Python đã bị xóa và bỏ bytecode cũ khi đổi bản Source.
for folder in ('core', 'cifar', 'experiments', 'kaggle'):
    for old in (SRC / folder).rglob('*.py'):
        if not (source_input / old.relative_to(SRC)).is_file():
            old.unlink()
for cached in SRC.rglob('*.pyc'):
    cached.unlink()
shutil.copytree(source_input, SRC, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', '.git', 'data', 'results'))
os.chdir(SRC)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
# Reload các module của project nếu chạy lại ô này sau khi đổi SOURCE_INPUT.
for module_name in list(sys.modules):
    if module_name in ('core', 'cifar', 'experiments') or module_name.startswith(('core.', 'cifar.', 'experiments.')):
        del sys.modules[module_name]
from experiments.exp32_cifar import build_parser, validate_args, plan_runs, child_command, RESUME_KEYS

print('Source:', source_input, '->', SRC)
print('Kết quả:', OUT)


## 2. Kế hoạch toàn bộ và nhóm chạy trong phiên

Ô này chỉ dựng danh sách cấu hình, **không tải CIFAR, không tạo model, không train**.
Run đã đủ 200 epoch sẽ được bỏ qua. Run dở được tiếp tục từ `last.pt` với cùng tên/cấu hình.
Bảng tiến độ ở phần sau luôn bao gồm cả lưới, kể cả nhóm chưa chọn trong phiên.


In [ ]:
ALL_DATASETS = ['cifar10', 'cifar100']
ALL_MODELS = ['preact_resnet18', 'wideresnet28_10', 'densenet_bc190']
ALL_METHODS = ['erm', 'mixup']
if not TARGET_SEEDS or any(type(s) is not int or s < 0 for s in TARGET_SEEDS):
    raise ValueError('TARGET_SEEDS cần là danh sách seed nguyên không âm, không rỗng.')
if len(set(TARGET_SEEDS)) != len(TARGET_SEEDS):
    raise ValueError('TARGET_SEEDS không được trùng.')
if type(WORKERS) is not int or WORKERS < 0:
    raise ValueError('WORKERS cần là số nguyên không âm.')
if MAX_SESSION_HOURS is not None and (not math.isfinite(MAX_SESSION_HOURS) or MAX_SESSION_HOURS <= 0):
    raise ValueError('MAX_SESSION_HOURS cần > 0 hoặc None.')
if REPORT_METRIC not in ('best_err', 'last_err', 'median_last10_err'):
    raise ValueError('REPORT_METRIC không hợp lệ.')

def select_values(values, available, label):
    selected = list(available) if values is None else list(values)
    if not selected or len(set(selected)) != len(selected) or any(v not in available for v in selected):
        raise ValueError(f'{label} phải là tập con không rỗng của {available}, hoặc None.')
    return selected

session_datasets = select_values(SESSION_DATASETS, ALL_DATASETS, 'SESSION_DATASETS')
session_models = select_values(SESSION_MODELS, ALL_MODELS, 'SESSION_MODELS')
session_methods = select_values(SESSION_METHODS, ALL_METHODS, 'SESSION_METHODS')
session_seeds = select_values(SESSION_SEEDS, TARGET_SEEDS, 'SESSION_SEEDS')
parser = build_parser()
args = parser.parse_args([
    '--suite', '--seeds', *map(str, TARGET_SEEDS), '--device', 'cuda',
    '--data-dir', str(DATA), '--out-dir', str(OUT), '--workers', str(WORKERS),
])
validate_args(parser, args)
ALL_RUNS = plan_runs(args)
assert len(ALL_RUNS) == 12 * len(TARGET_SEEDS)
assert all(r.paper_protocol and r.epochs == 200 and not r.amp for r in ALL_RUNS)

def method_of(run):
    return 'erm' if run.alpha == 0 else 'mixup'

def in_session(run):
    return (run.dataset in session_datasets and run.model in session_models
            and method_of(run) in session_methods and run.seed in session_seeds)

SESSION_RUNS = [r for r in ALL_RUNS if in_session(r)]
plan_payload = {
    'created_utc': datetime.now(timezone.utc).isoformat(), 'source_input': str(source_input),
    'target_seeds': TARGET_SEEDS, 'session_run_names': [r.run_name for r in SESSION_RUNS],
    'runs': [vars(r) for r in ALL_RUNS],
}
(MANAGEMENT / 'plan.json').write_text(json.dumps(plan_payload, indent=2), encoding='utf-8')
print(f'Toàn bộ: {len(ALL_RUNS)} runs | Chọn trong phiên: {len(SESSION_RUNS)} runs')
display(pd.DataFrame([
    {'dataset': r.dataset, 'model': r.model, 'method': method_of(r), 'seed': r.seed,
     'epochs': r.epochs, 'LR milestones': r.milestones, 'trong phiên': in_session(r)}
    for r in ALL_RUNS
]))


## 3. Khôi phục/gom kết quả từ các phiên trước

Trong `RESUME_INPUTS`, điền **thư mục `results`**, không phải `results_logs.zip`.
Notebook chỉ xét các run thuộc lưới hiện tại, kiểm tra cấu hình và lịch sử chung rồi chọn
snapshot có nhiều epoch đã lưu hơn. Snapshot có cùng tên nhưng khác cấu hình hoặc khác
lịch sử sẽ báo lỗi; hãy bỏ input không thuộc cùng một chuỗi chạy hoặc quản lý nó riêng.
Kaggle Input được giữ nguyên. Nếu output cũ cũng chứa code `Source`, hãy đặt
`SOURCE_INPUT` rõ ràng để chọn bản code muốn chạy.

Tiến độ run dở được đọc từ **checkpoint**, vì CSV có thể đã ghi thêm một dòng ngay trước
khi phiên bị ngắt. Bản CSV trong working được đồng bộ theo lịch sử đã lưu trong checkpoint.
Run hoàn tất chỉ có log vẫn dùng để tổng hợp; run dở cần `last.pt`.

Resume khôi phục trạng thái trainer đã lưu, nhưng DataLoader/worker augmentation chưa được
lưu đầy đủ trong trainer hiện tại: kết quả có thể khác một lần huấn luyện liên tục.


In [ ]:
def read_history(path):
    if not path.is_file():
        return []
    with path.open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def history_signature(rows):
    # Metrics của trainer đều là số; CSV và checkpoint được so ở cùng biểu diễn.
    return [tuple(sorted((key, float(value)) for key, value in row.items())) for row in rows]

def validate_history(rows, label):
    if [int(row['epoch']) for row in rows] != list(range(len(rows))):
        raise ValueError(f'History thiếu/trùng epoch: {label}')

def inspect_snapshot(run, directory):
    if not directory.exists() or not any(directory.iterdir()):
        return {'epoch': 0, 'complete': False, 'checkpoint': False, 'history': [], 'exists': False}
    config_path = directory / 'config.json'
    if not config_path.is_file():
        raise ValueError(f'Thiếu config.json: {directory}')
    saved = json.loads(config_path.read_text(encoding='utf-8'))['config']
    saved.setdefault('model', 'preact_resnet18')
    saved.setdefault('test_batch_size', 100)
    mismatches = [key for key in RESUME_KEYS if saved.get(key) != getattr(run, key)]
    if mismatches or saved.get('epochs', 0) > run.epochs:
        raise ValueError(f'Cấu hình khác kế hoạch ({mismatches}): {directory}')
    rows = read_history(directory / 'history.csv')
    summary_path = directory / 'summary.json'
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else {}
    except json.JSONDecodeError:
        summary = {}
        if not (directory / 'last.pt').is_file():
            raise ValueError(f'Summary bị ghi dở và thiếu last.pt: {directory}')
    has_checkpoint = (directory / 'last.pt').is_file()
    if (summary.get('epochs_run') == run.epochs and len(rows) == run.epochs
            and saved.get('epochs') == run.epochs and saved.get('paper_protocol') is True):
        validate_history(rows, directory)
        return {'epoch': run.epochs, 'complete': True, 'checkpoint': has_checkpoint,
                'history': rows, 'exists': True}
    if not has_checkpoint:
        if rows or summary:
            raise ValueError(f'Run dở thiếu last.pt, không thể resume: {directory}')
        return {'epoch': 0, 'complete': False, 'checkpoint': False, 'history': [], 'exists': True}
    # Chỉ dùng checkpoint do mình/nhóm tạo. Không gọi core.load_checkpoint ở đây vì hàm đó đổi RNG.
    checkpoint = torch.load(directory / 'last.pt', map_location='cpu', weights_only=False)
    state = checkpoint['state']
    epoch, history = state['epoch'], state['history']
    del checkpoint
    validate_history(history, directory / 'last.pt')
    if type(epoch) is not int or epoch != len(history) or not 0 <= epoch <= run.epochs:
        raise ValueError(f'Checkpoint có epoch/history không hợp lệ: {directory}')
    # Chấp nhận CSV thiếu/dư phần đuôi, nhưng phần chung phải thuộc cùng một trajectory.
    common = min(len(rows), len(history))
    if history_signature(rows[:common]) != history_signature(history[:common]):
        raise ValueError(f'CSV và checkpoint khác lịch sử: {directory}')
    return {'epoch': epoch, 'complete': False, 'checkpoint': True,
            'history': history, 'exists': True}

def write_history(path, rows):
    if not rows:
        path.unlink(missing_ok=True)
        return
    temporary = path.with_suffix('.csv.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)

def copy_snapshot(source, destination):
    # Sao chép cả snapshot; không ghép last.pt của run này với summary/best của run khác.
    if not destination.resolve().is_relative_to(OUT.resolve()):
        raise ValueError('Đích restore phải nằm dưới results của working.')
    shutil.copytree(source, destination, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.tmp'))
    for name in ('summary.json', 'last.pt', 'best_model.pt', 'final_model.pt'):
        stale = destination / name
        if not (source / name).exists() and stale.is_file():
            stale.unlink()

resume_roots = []
for entry in RESUME_INPUTS:
    root = Path(entry).resolve()
    if not root.is_relative_to(INPUT.resolve()) or not root.is_dir():
        raise ValueError(f'RESUME_INPUTS cần thư mục results có thật trong /kaggle/input: {root}')
    if root not in resume_roots:
        resume_roots.append(root)

# Kiểm tra tất cả input trước khi ghi các snapshot được chọn vào working.
winners = {}
for run in ALL_RUNS:
    local = OUT / run.run_name
    choices = [(local, inspect_snapshot(run, local))]
    for root in resume_roots:
        candidate = root / run.run_name
        if candidate.is_dir():
            choices.append((candidate, inspect_snapshot(run, candidate)))
    best_path, best = choices[0]
    for path, info in choices[1:]:
        common = min(best['epoch'], info['epoch'])
        if history_signature(best['history'][:common]) != history_signature(info['history'][:common]):
            raise ValueError(f'Hai snapshot khác lịch sử: {best_path} và {path}')
        rank = lambda item: (item['epoch'], item['checkpoint'], item['complete'])
        if rank(info) > rank(best):
            best_path, best = path, info
    winners[run.run_name] = (best_path, best)

RUN_STATES = {}
for run in ALL_RUNS:
    path, info = winners[run.run_name]
    destination = OUT / run.run_name
    if path != destination:
        copy_snapshot(path, destination)
        print('Khôi phục:', run.run_name, '| epoch đã lưu:', info['epoch'])
    if info['checkpoint'] and not info['complete']:
        write_history(destination / 'history.csv', info['history'])
    RUN_STATES[run.run_name] = info
print('Đã kiểm tra/khôi phục kết quả; không thay đổi Kaggle Input.')


## 4. Theo dõi tiến độ

`epoch đã lưu` là số epoch đã hoàn tất (0–200), không phải chỉ số epoch bắt đầu từ 0 trong CSV.
Checklist liệt kê các seed chưa hoàn tất cho từng dataset/model/phương pháp.


In [ ]:
def save_progress():
    rows = []
    for run in ALL_RUNS:
        state = RUN_STATES[run.run_name]
        status = 'hoàn tất' if state['complete'] else ('cần resume' if state['checkpoint'] else 'chưa chạy')
        rows.append({'dataset': run.dataset, 'model': run.model, 'method': method_of(run),
                     'seed': run.seed, 'epoch đã lưu': state['epoch'], 'mục tiêu': run.epochs,
                     'trạng thái': status, 'lỗi gần nhất': state.get('error', ''),
                     'trong phiên': in_session(run), 'run_name': run.run_name})
    progress = pd.DataFrame(rows)
    progress.to_csv(MANAGEMENT / 'progress.csv', index=False)
    missing_rows = []
    for dataset in ALL_DATASETS:
        for model in ALL_MODELS:
            for method in ALL_METHODS:
                group = progress[(progress.dataset == dataset) & (progress.model == model)
                                 & (progress.method == method)]
                done = group.loc[group['trạng thái'] == 'hoàn tất', 'seed'].tolist()
                missing_rows.append({'dataset': dataset, 'model': model, 'method': method,
                                     'seed hoàn tất': done, 'seed còn thiếu': [s for s in TARGET_SEEDS if s not in done]})
    missing = pd.DataFrame(missing_rows)
    missing.to_csv(MANAGEMENT / 'missing_runs.csv', index=False)
    print(f"Hoàn tất {sum(progress['trạng thái'] == 'hoàn tất')}/{len(progress)} runs")
    display(progress)
    display(missing)
    return progress

progress = save_progress()


## 5. Chạy nhóm đã chọn

Mỗi run dùng một tiến trình riêng; log được in ngay và lưu trong `results/management/run_logs/`.
Lỗi được lưu trong nhật ký phiên. Mặc định dừng hàng đợi khi có lỗi để các ô báo cáo/đóng gói vẫn chạy.
Giữ nguyên batch/FP32 khi so với protocol mặc định; nếu thiếu VRAM, chọn GPU đủ bộ nhớ hoặc
thực hiện biến thể và ghi rõ trong báo cáo.

`MAX_SESSION_HOURS` tính từ lúc bắt đầu ô chạy train và chỉ **ngăn bắt đầu run tiếp theo**.
Run đang chạy có thể vượt ngân sách này; cần chừa thời gian cho nó kết thúc và Kaggle lưu output.
Dừng/interrupt sẽ kết thúc tiến trình con; epoch chưa lưu cần chạy lại khi resume.

`HARD_STOP_HOURS` (mặc định 11 giờ, tính từ lúc mở notebook) **dừng hẳn run đang chạy** rồi để các ô
báo cáo/đóng gói chạy tiếp, nhờ đó version luôn lưu được `last.pt` trước khi Kaggle cắt phiên ở 12 giờ.
Phiên sau chỉ cần Add Input output version này và điền `RESUME_INPUTS`.


In [ ]:
class HardStop(Exception):
    """Run bị dừng có chủ ý vì sắp hết 12 giờ của phiên Kaggle; resume ở phiên sau."""


def _stop_process(process):
    if process.poll() is not None:
        return
    if os.name == 'nt':
        process.terminate()
    else:
        os.killpg(process.pid, signal.SIGTERM)
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        if os.name == 'nt':
            process.kill()
        else:
            os.killpg(process.pid, signal.SIGKILL)
        process.wait()


def stream_command(command, log_path, deadline=None):
    """deadline: mốc time.monotonic(); quá mốc thì dừng tiến trình con và báo HardStop.
    last.pt được lưu nguyên tử sau mỗi epoch, nên chỉ mất tối đa epoch đang chạy dở."""
    import threading
    print('$', shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n$ ' + shlex.join(command) + '\n')
        log.flush()
        process = subprocess.Popen(command, cwd=SRC, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, start_new_session=(os.name != 'nt'),
                                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        timed_out = threading.Event()
        watchdog = None
        if deadline is not None:
            def _on_deadline():
                timed_out.set()
                _stop_process(process)
            watchdog = threading.Timer(max(0.0, deadline - time.monotonic()), _on_deadline)
            watchdog.daemon = True
            watchdog.start()
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            rc = process.wait()
            if timed_out.is_set():
                raise HardStop(f'Hết HARD_STOP_HOURS; đã dừng run để lưu output.')
            return rc
        except HardStop:
            raise
        except BaseException:
            if process.poll() is None:
                if os.name == 'nt':
                    process.terminate()
                else:
                    os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                if os.name == 'nt':
                    process.kill()
                else:
                    os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
        finally:
            if watchdog is not None:
                watchdog.cancel()
            process.stdout.close()

session_events = []
session_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
events_path = MANAGEMENT / f'session_{session_id}.json'

def record_event(event):
    session_events.append(event)
    events_path.write_text(json.dumps(session_events, ensure_ascii=False, indent=2), encoding='utf-8')

if not RUN_TRAINING:
    print('Đang xem kế hoạch: đổi RUN_TRAINING=True ở ô cấu hình và chạy lại để huấn luyện.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('Cần GPU CUDA để train.')
    ready = True
    if RUN_TESTS:
        rc = stream_command([sys.executable, '-m', 'pytest', 'tests', '-q'], MANAGEMENT / 'tests.log')
        record_event({'action': 'tests', 'exit_code': rc})
        ready = rc == 0
        if not ready:
            print('Tests lỗi; không bắt đầu huấn luyện. Xem management/tests.log.')
    session_start = time.monotonic()
    hard_deadline = None if HARD_STOP_HOURS is None else NOTEBOOK_START + HARD_STOP_HOURS * 3600
    for run in SESSION_RUNS if ready else []:
        if RUN_STATES[run.run_name]['complete']:
            print('Bỏ qua run hoàn tất:', run.run_name)
            continue
        if hard_deadline is not None and time.monotonic() >= hard_deadline:
            record_event({'action': 'hard_stop', 'next_run': run.run_name})
            print('Đã tới HARD_STOP_HOURS; các run còn lại để phiên sau.')
            break
        if MAX_SESSION_HOURS is not None and time.monotonic() - session_start >= MAX_SESSION_HOURS * 3600:
            record_event({'action': 'budget_stop', 'next_run': run.run_name})
            print('Đã hết ngân sách phiên; các run còn lại được giữ cho phiên sau.')
            break
        started = time.monotonic()
        event = {'run_name': run.run_name, 'started_utc': datetime.now(timezone.utc).isoformat()}
        record_event({**event, 'action': 'start'})
        try:
            # Kiểm tra lại nếu người dùng vừa chép kết quả vào working sau ô restore.
            state = inspect_snapshot(run, OUT / run.run_name)
            if state['checkpoint'] and not state['complete']:
                write_history(OUT / run.run_name / 'history.csv', state['history'])
            rc = stream_command(child_command(run), MANAGEMENT / 'run_logs' / (run.run_name + '.log'),
                                deadline=hard_deadline)
            state = inspect_snapshot(run, OUT / run.run_name)
            RUN_STATES[run.run_name] = state
            if rc != 0 or not state['complete']:
                raise RuntimeError(f'Run chưa hoàn tất, exit={rc}. Xem log và giữ last.pt để resume.')
        except HardStop as stop:
            record_event({**event, 'action': 'hard_stop', 'wall_time_s': time.monotonic() - started})
            RUN_STATES[run.run_name] = inspect_snapshot(run, OUT / run.run_name)
            print(stop, '| epoch đã lưu:', RUN_STATES[run.run_name]['epoch'],
                  '| phiên sau: Add Input output version này và điền RESUME_INPUTS.')
            break
        except KeyboardInterrupt:
            record_event({**event, 'action': 'interrupted', 'wall_time_s': time.monotonic() - started})
            try:
                RUN_STATES[run.run_name] = inspect_snapshot(run, OUT / run.run_name)
            except Exception as snapshot_error:
                RUN_STATES[run.run_name]['error'] = str(snapshot_error)
            print('Đã dừng tiến trình train; tiếp tục các ô dưới để lưu báo cáo/output.')
            break
        except Exception as error:
            record_event({**event, 'action': 'failed', 'error': str(error),
                          'wall_time_s': time.monotonic() - started})
            try:
                RUN_STATES[run.run_name] = inspect_snapshot(run, OUT / run.run_name)
            except Exception:
                pass
            RUN_STATES[run.run_name]['error'] = str(error)
            print('LỖI:', run.run_name, '|', error)
            if STOP_ON_ERROR:
                break
        else:
            record_event({**event, 'action': 'complete', 'wall_time_s': time.monotonic() - started})
    print('Nhật ký phiên:', events_path)
progress = save_progress()


## 6. Bảng đối chiếu paper và đường lỗi theo epoch

Chỉ nhận run hoàn tất đúng protocol 200 epoch; run còn thiếu không được điền bằng số paper.
Report tính mean/sample std qua các seed đã có và hiện số run. Nếu chưa đủ toàn bộ seed,
đây là **báo cáo tạm thời**; xem `missing_runs.csv` để biết phần còn thiếu.

Mục 3.2 không nói rõ quy tắc chọn checkpoint. `best_err` là lựa chọn báo cáo của repo;
có thể đổi `REPORT_METRIC` sang `last_err` hoặc `median_last10_err` để đối chiếu.
`training_curves.png` gồm sáu ô dataset/model, trong đó ô DenseNet/CIFAR-10 có hai đường
ERM/mixup tương ứng nội dung Figure 3(b). Bảng đối chiếu tương ứng Figure 3(a).


In [ ]:
from experiments.exp32_report import main as make_report

completed = [r for r in ALL_RUNS if RUN_STATES[r.run_name]['complete']]
if completed:
    # Chỉ đưa các seed/run mục tiêu vào report; staging ở ngoài results để tránh đếm trùng.
    with tempfile.TemporaryDirectory(prefix='exp32_report_', dir=WORK) as staging:
        report_input = Path(staging)
        for run in completed:
            destination = report_input / run.run_name
            destination.mkdir(parents=True, exist_ok=True)
            for name in ('config.json', 'history.csv', 'summary.json'):
                shutil.copy2(OUT / run.run_name / name, destination / name)
        report = make_report([str(report_input), '--metric', REPORT_METRIC,
                              '--output-dir', str(OUT / 'report'), '--plot'])
    # Lưu đường dẫn run gốc thay vì thư mục staging đã xóa.
    report['run_directories'] = [str(OUT / r.run_name) for r in completed]
    (OUT / 'report/comparison.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    comparison = pd.read_csv(OUT / 'report/comparison.csv')
    display(comparison[['dataset', 'model', 'method', 'count', 'seeds', 'mean_err',
                        'sample_std_err', 'paper_err', 'gap_to_paper']])
    display(Image(filename=str(OUT / 'report/training_curves.png')))
    print(f'Báo cáo dựa trên {len(completed)}/{len(ALL_RUNS)} runs mục tiêu đã hoàn tất.')
else:
    # Bỏ báo cáo của lưới cũ nếu người dùng đã đổi TARGET_SEEDS trong cùng working.
    for name in ('comparison.csv', 'comparison.json', 'training_curves.png'):
        (OUT / 'report' / name).unlink(missing_ok=True)
    print('Chưa có run đủ 200 epoch để đối chiếu. Xem management/progress.csv và missing_runs.csv.')


## 7. Lưu kết quả để tải về và tiếp tục ở phiên sau

- **`results_logs.zip`**: config, CSV, JSON, log và biểu đồ; loại trọng số `.pt` và file `.tmp`.
- **`results/` trong Output của version**: giữ `last.pt` (resume), `best_model.pt`, `final_model.pt`.
  Snapshot có checkpoint lớn hơn zip log; kiểm tra dung lượng Output sau khi lưu version.
- **`management/source_used/`**: bản code đã dùng, đi kèm kế hoạch và nhật ký phiên.

Để tiếp tục: lưu version có output, Add Input output đó vào phiên mới, thêm đường dẫn
`/kaggle/input/<output-đã-gắn>/results` vào `RESUME_INPUTS`, chọn nhóm `SESSION_*` tiếp theo
và chạy lại notebook. Có thể gom nhiều output qua `RESUME_INPUTS` để tạo báo cáo chung.


In [ ]:
source_snapshot = MANAGEMENT / 'source_used'
for folder in ('core', 'cifar', 'experiments', 'kaggle'):
    shutil.copytree(SRC / folder, source_snapshot / folder, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache'))
for name in ('requirements.txt', 'environment.yml'):
    if (SRC / name).is_file():
        shutil.copy2(SRC / name, source_snapshot / name)

archive = WORK / 'results_logs.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(OUT.rglob('*')):
        if (path.is_file() and path.suffix not in ('.pt', '.tmp')
                and '__pycache__' not in path.parts):
            zipped.write(path, path.relative_to(WORK))
size = sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 2**30
print('Zip log:', archive, '|', round(archive.stat().st_size / 2**20, 2), 'MiB')
print('Thư mục results gồm checkpoint:', round(size, 2), 'GiB')
print('Giữ results trong Output của version để dùng RESUME_INPUTS ở phiên sau.')
